# Re-evaluate published models on the corrected val/test split

`src/data.py`'s `FlowerDataModule.setup()` had a bug: `val_set` and `test_set` were both built from `train_subset`, so every reported `val_acc`/`val_f1` (including the two models pushed to the HF Hub, both logged at ~1.0) was measured on training data. The split logic is now fixed (val/test use their own subsets). This notebook reloads the two published checkpoints and re-scores them on the real, disjoint val and test sets.

In [1]:
import sys
from pathlib import Path

import lightning as L

BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR))
sys.path.insert(0, str(BASE_DIR / "train"))

from train.config import PRETRAINED_MODEL_REGISTRY  # noqa: E402
from src.classifier import FlowerClassifier  # noqa: E402
from src.data import FlowerDataModule, FlowerDataset  # noqa: E402

CKPT_DIR = BASE_DIR / "models_checkpoints"
DATA_ROOT = BASE_DIR / "data"

# The two checkpoints published to the HF Hub (see notebooks/publish_to_hf.ipynb)
PUBLISHED_MODELS = [
    {
        "pretrained_model": "vit_b_16",
        "ckpt": "vit_b_16-epoch=18-val_acc=1.000-6899e8e2.ckpt",
    },
    {
        "pretrained_model": "efficientnet_v2_s",
        "ckpt": "efficientnet_v2_s-epoch=25-val_acc=1.000-d6508546.ckpt",
    },
]

In [2]:
dataset = FlowerDataset(DATA_ROOT)
num_classes = len(dataset.classes)

# same seed/split ratios FlowerDataModule used during training (train/run_training.py
# only overrides data_root and batch_size), so this reproduces the intended val/test sets
dm = FlowerDataModule(DATA_ROOT, batch_size=32)
dm.setup("fit")
dm.setup("test")

print(f"train={len(dm.train_set)} val={len(dm.val_set)} test={len(dm.test_set)}")

train=5733 val=1228 test=1228


In [3]:
def load_model(pretrained_model_name: str, ckpt_name: str) -> FlowerClassifier:
    factory, head_name, _ = PRETRAINED_MODEL_REGISTRY[pretrained_model_name]
    model = FlowerClassifier.load_from_checkpoint(
        CKPT_DIR / ckpt_name,
        pretrained_model=factory(),
        num_classes=num_classes,
        class_weights=None,
        head_name=head_name,
        class_names=dataset.classes,
        map_location="cpu",
        strict=False,  # checkpoint has a criterion.weight buffer we don't restore for inference
    )
    model.eval()
    return model

In [4]:
results = []
trainer = L.Trainer(
    accelerator="auto", devices=1, logger=False, enable_checkpointing=False
)

for spec in PUBLISHED_MODELS:
    model = load_model(spec["pretrained_model"], spec["ckpt"])
    val_metrics = trainer.validate(model, datamodule=dm, verbose=False)[0]
    test_metrics = trainer.test(model, datamodule=dm, verbose=False)[0]
    results.append(
        {"architecture": spec["pretrained_model"], **val_metrics, **test_metrics}
    )
    del model

results

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/zelluzy/Desktop/code/flowers/.venv/lib/python3.13/site-packages/lightning/pytorch/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['criterion.weight']
You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/zelluzy/Desktop/code/flowers/.venv/lib/python3.13/site-packages/lightning/pyto

Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

[{'architecture': 'vit_b_16',
  'val_loss': 0.10145033895969391,
  'val_acc': 0.9796416759490967,
  'val_f1': 0.9658452868461609,
  'test_loss': 0.0972985103726387,
  'test_acc': 0.9796416759490967,
  'test_f1': 0.9636799097061157},
 {'architecture': 'efficientnet_v2_s',
  'val_loss': 0.12455179542303085,
  'val_acc': 0.9682410359382629,
  'val_f1': 0.9468428492546082,
  'test_loss': 0.12059381604194641,
  'test_acc': 0.9641693830490112,
  'test_f1': 0.9364289045333862}]

In [5]:
import pandas as pd

comparison = pd.DataFrame(results).set_index("architecture")
comparison.to_csv("published_models_corrected_eval.csv")
comparison

,val_loss,val_acc,val_f1,test_loss,test_acc,test_f1
architecture,,,,,,
vit_b_16,0.101450,0.979642,0.965845,0.097299,0.979642,0.963680
efficientnet_v2_s,0.124552,0.968241,0.946843,0.120594,0.964169,0.936429
